In [5]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# import pandas as pd
# import numpy as np

# # =========================
# # 경로 설정 (비워둠)
# # =========================
# DF2_PATH = r"C:/Users/qkrtl/Downloads/df2.csv"                  # 예: r"C:\...\df2.csv"
# MASTER_PATH = r"C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output/master_all_columns.csv"               # 예: r"C:\...\master_all_columns.csv"


# # =========================
# # 조인 키 (기본: matchId + name)
# # =========================
# KEYS = ["matchId", "name"]

# # survival_time 컬럼 후보 (master에서 찾기)
# SURVIVAL_CANDIDATES = ["survival_time", "survivalTime", "timeAlive", "TimeAlive"]
# SURVIVAL_THRESHOLD = 120  # 2분

# # =========================
# # 유틸
# # =========================
# def normalize_key_values(df: pd.DataFrame, keys):
#     out = df.copy()
#     for k in keys:
#         out[k] = out[k].astype("string").str.strip().replace("", pd.NA)
#     return out

# def pick_survival_col(df: pd.DataFrame, candidates):
#     for c in candidates:
#         if c in df.columns:
#             return c
#     return None

# # =========================
# # 1) 로드
# # =========================
# df2 = pd.read_csv(DF2_PATH, low_memory=False)
# master = pd.read_csv(MASTER_PATH, low_memory=False)

# # 키 존재 체크
# for k in KEYS:
#     if k not in df2.columns:
#         raise KeyError(f"df2에 키 컬럼이 없습니다: {k}")
#     if k not in master.columns:
#         raise KeyError(f"master_all_columns에 키 컬럼이 없습니다: {k}")

# # 키 정규화(공백/타입)
# df2_n = normalize_key_values(df2, KEYS)
# master_n = normalize_key_values(master, KEYS)

# # =========================
# # 2) master 쪽 중복키 처리 (중요)
# #    - left join에서 master에 같은 키가 여러 개면 df2 행이 증식함
# # =========================
# dup_master = master_n.duplicated(KEYS).sum()
# print(f"[master] duplicated keys rows = {dup_master:,}")

# if dup_master > 0:
#     # 기본 정책: 같은 키면 "첫 번째"만 사용
#     # (원하면 '마지막' 또는 특정 컬럼 기준 정렬 후 drop_duplicates로 바꿔도 됨)
#     master_n = master_n.drop_duplicates(KEYS, keep="first").copy()
#     print(f"[master] after dedup, rows = {len(master_n):,}")

# # =========================
# # 3) df2 기준 LEFT JOIN
# #    - 겹치는 컬럼명은 _df2 / _master suffix로 구분
# # =========================
# merged = df2_n.merge(
#     master_n,
#     on=KEYS,
#     how="left",
#     suffixes=("_df2", "_master"),
#     indicator=True,            # 매칭 여부 확인용
#     validate="m:1"             # df2 many : master one (dedup 했으므로 이게 정상)
# )

# # =========================
# # 4) 매칭 리포트
# # =========================
# vc = merged["_merge"].value_counts(dropna=False)
# print("\n[JOIN 결과 _merge 카운트]")
# print(vc.to_string())

# match_rate = (merged["_merge"] == "both").mean()
# print(f"\n[JOIN 매칭률] {match_rate:.2%} (df2 기준)")

# # =========================
# # 5) survival_time 기반 플래그 컬럼 생성 (필터 대신 컬럼으로 관리)
# # =========================
# surv_col = pick_survival_col(merged, SURVIVAL_CANDIDATES)
# print(f"\n[사용 survival 컬럼] {surv_col}")

# if surv_col is not None:
#     merged[surv_col] = pd.to_numeric(merged[surv_col], errors="coerce")
#     merged["is_2min_over"] = merged[surv_col].ge(SURVIVAL_THRESHOLD)
#     merged["is_2min_under"] = merged[surv_col].lt(SURVIVAL_THRESHOLD)
#     merged["survival_is_na"] = merged[surv_col].isna()
# else:
#     # master에서 survival_time이 안 붙은 경우(매칭 실패 포함) 대비
#     merged["is_2min_over"] = pd.NA
#     merged["is_2min_under"] = pd.NA
#     merged["survival_is_na"] = pd.NA

# # =========================
# # 6) 필요하면 indicator 컬럼 제거
# # =========================
# # merged = merged.drop(columns=["_merge"])

# # =========================
# # 7) 저장(원하면)
# # =========================
# OUT_PATH = r"C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output/master.csv" 
# merged.to_csv(OUT_PATH, index=False)
# print("\n완료: df2 LEFT JOIN master_all_columns + is_2min_over 생성")


[master] duplicated keys rows = 28,045
[master] after dedup, rows = 823,389

[JOIN 결과 _merge 카운트]
_merge
both          821596
left_only       6505
right_only         0

[JOIN 매칭률] 99.21% (df2 기준)

[사용 survival 컬럼] survival_time

완료: df2 LEFT JOIN master_all_columns + is_2min_over 생성


In [6]:
df = pd.read_csv("C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output/master.csv")
df.head()


,matchId,mapName_df2,gameMode_df2,createdAt_df2,DBNOs_df2,assists_df2,boosts,damageDealt,deathType_df2,headshotKills_df2,...,weaponsAcquired_master,winPlace_df2,servername_master,currentRankPoint_master,game_type_master,tier_master,_merge,is_2min_over,is_2min_under,survival_is_na
0,6f3b4b92-5c17-4a7e-a8b0-4a35ad7c1d7a,Desert_Main,squad,2026-02-15 18:31:26+00:00,2,0,0,200.00000,byplayer,0,...,8.0,18.0,steam,801.0,official,Bronze,both,True,False,False
1,fcf230a3-e9bd-4c7e-a5ac-28e998a499d5,Desert_Main,squad,2026-02-19 06:03:34+00:00,2,0,3,203.72414,byplayer,2,...,9.0,10.0,steam,1910.0,competitive,Gold,both,True,False,False
2,d76c3295-4742-467e-9d86-c58edfebcdcf,Neon_Main,squad,2026-02-11 20:02:02+00:00,1,4,6,451.97598,byplayer,2,...,10.0,7.0,steam,0.0,official,Unranked,both,True,False,False
3,58878e90-c3c3-4be9-bd42-268f3a7ee3e2,Tiger_Main,squad,2026-02-11 22:08:41+00:00,2,1,5,382.21080,byplayer,0,...,13.0,4.0,steam,0.0,official,Unranked,both,True,False,False
4,4bad3d1a-2bf0-4921-8b14-50984cbdf2eb,Baltic_Main,squad,2026-02-16 14:36:19+00:00,3,0,7,428.00513,byplayer,1,...,16.0,8.0,steam,0.0,official,Unranked,both,True,False,False


In [7]:
df.shape

(828101, 101)

In [10]:
df2 = pd.read_csv("C:/Users/qkrtl/Downloads/df2.csv")

erangel = pd.read_parquet("C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output/erangel_clustered.parquet")
miramar = pd.read_parquet("C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output/miramar_clustered.parquet")
rondo   = pd.read_parquet("C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output/rondo_clustered.parquet")
taego   = pd.read_parquet("C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output/taego_clustered.parquet")

clustered_df = pd.concat([erangel, miramar, rondo, taego], ignore_index=True)

merged = pd.merge(
    df2,
    clustered_df,
    left_on=["matchId", "playerId"],
    right_on=["matchId", "accountId"],
    how="left"
)

In [11]:
merged.to_csv("master.csv")

In [13]:
merged.shape

(828101, 68)

In [15]:
merged.T

,0,1,2,3,4,5,6,7,8,9,...,828091,828092,828093,828094,828095,828096,828097,828098,828099,828100
matchId,6f3b4b92-5c17-4a7e-a8b0-4a35ad7c1d7a,fcf230a3-e9bd-4c7e-a5ac-28e998a499d5,d76c3295-4742-467e-9d86-c58edfebcdcf,58878e90-c3c3-4be9-bd42-268f3a7ee3e2,4bad3d1a-2bf0-4921-8b14-50984cbdf2eb,af957311-c148-4b8d-9708-396ec41f9dd9,51656ac0-1e85-419d-ac1e-27287c81995c,6f6c31c7-63be-416b-95dd-42d89dffccdc,5bbf91fc-6cac-4d42-89f2-de11c4e6d28a,9c1beb47-6956-491d-9907-d982ee65a478,...,220d4efc-5604-40f0-ba7a-3b2b855ee245,acdfc4e4-e657-4c4f-859b-5c2e8546933e,f5de6cff-df82-4097-bb9f-a39d1bb20176,66da2633-4bab-4bd5-92cd-9ad73098acbb,a5b37d26-e679-4f70-8912-dbc0c9e72896,5c29f5c3-d0ea-457d-b2b8-0f78df6a0ada,29e661b4-f8d1-4d57-b1d3-2dd44753a752,60e703c6-589a-4a67-85f3-b5a31ade82ed,0371162c-d278-4ad2-8fc1-f07985991e72,e9588a80-5c13-4e22-8496-90bc9429c928
mapName,Desert_Main,Desert_Main,Neon_Main,Tiger_Main,Baltic_Main,Tiger_Main,Tiger_Main,Tiger_Main,Tiger_Main,Neon_Main,...,Neon_Main,Tiger_Main,Tiger_Main,Tiger_Main,Tiger_Main,Tiger_Main,Baltic_Main,Neon_Main,Baltic_Main,Tiger_Main
gameMode,squad,squad,squad,squad,squad,squad,squad,squad,squad-fpp,squad-fpp,...,squad,squad,squad,squad,squad,squad,squad,squad,squad,squad
createdAt,2026-02-15 18:31:26+00:00,2026-02-19 06:03:34+00:00,2026-02-11 20:02:02+00:00,2026-02-11 22:08:41+00:00,2026-02-16 14:36:19+00:00,2026-02-12 15:45:04+00:00,2026-02-17 06:20:29+00:00,2026-02-24 13:43:01+00:00,2026-02-17 00:13:08+00:00,2026-02-17 00:35:17+00:00,...,2026-02-14 02:40:42+00:00,2026-02-24 20:35:45+00:00,2026-02-13 14:59:13+00:00,2026-02-17 11:41:45+00:00,2026-02-19 14:11:23+00:00,2026-02-19 01:42:21+00:00,2026-02-24 12:23:16+00:00,2026-02-13 13:53:48+00:00,2026-02-13 06:27:49+00:00,2026-02-20 13:29:16+00:00
DBNOs,2,2,1,2,3,4,4,1,0,3,...,4,0,0,0,2,2,0,0,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
score_C2_aggro,0.336165,2.101374,2.508123,3.668261,-0.045901,-0.456073,0.596656,2.901525,0.322327,-0.869627,...,0.57705,-0.306039,-0.107261,0.927659,-1.216028,0.046296,1.199585,2.209888,7.872853,0.720648
score_C3_guerilla,-1.645525,-0.960831,-1.776159,0.168623,1.682772,0.533029,1.913537,-0.590179,-1.686227,-0.687904,...,2.067349,-0.252934,-0.244331,0.0,1.715904,-0.216918,-1.319399,-1.011736,-3.035208,1.28
umap_x,NaN,NaN,NaN,NaN,-1.380472,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.168689,NaN,0.116763,NaN
umap_y,NaN,NaN,NaN,NaN,9.474584,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,16.354538,NaN,-0.236922,NaN


In [16]:
merged['persona']

0          🏃 중앙 점령형
1         ⚔️ 중앙 교전형
2         ⚔️ 중앙 교전형
3         ⚔️ 중앙 교전형
4         🦅 게릴라 운영형
            ...    
828096     🌿 외곽 운영형
828097     🏃 중앙 점령형
828098    ⚔️ 중앙 교전형
828099    ⚔️ 중앙 교전형
828100    🦅 게릴라 운영형
Name: persona, Length: 828101, dtype: str

In [19]:
merged['persona'].unique()

<ArrowStringArray>
['🏃 중앙 점령형', '⚔️ 중앙 교전형', '🦅 게릴라 운영형', '🌿 외곽 운영형', '❓ 혼합형', nan]
Length: 6, dtype: str

In [21]:
merged.describe()

,DBNOs,assists,boosts_x,damageDealt_x,headshotKills,heals_x,killPlace,killStreaks,kills_x,longestKill,...,move_speed,heal_boost_use,margin,score_C0_center,score_C1_edge,score_C2_aggro,score_C3_guerilla,umap_x,umap_y,margin_threshold
count,828101.000000,828101.000000,828101.000000,828101.000000,828101.000000,828101.000000,828101.000000,828101.000000,828101.000000,828101.000000,...,821596.000000,821596.000000,821596.000000,821596.000000,821596.000000,821596.000000,821596.000000,212283.000000,212283.000000,6.093130e+05
mean,1.057735,0.492252,3.627338,184.417603,0.256088,2.400394,35.226853,0.588997,1.016417,35.169157,...,1.973541,6.027052,1.159393,0.463658,0.219053,0.587920,-0.390829,0.459794,7.666155,2.000000e-01
std,1.415820,0.909413,3.407429,203.878250,0.596276,3.615120,22.259332,0.730785,1.561216,70.953750,...,0.664178,5.938032,93.717876,1.427078,93.745348,1.770636,1.816401,3.374259,4.844881,2.775560e-17
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,-8.204323,-9.089759,-3.969275,-15.766222,-5.454636,-3.729967,2.000000e-01
25%,0.000000,0.000000,0.000000,35.528084,0.000000,0.000000,17.000000,0.000000,0.000000,0.000000,...,1.580229,1.000000,0.298138,-0.168623,-1.180359,-0.604955,-1.469170,-2.073887,5.409525,2.000000e-01
50%,1.000000,0.000000,3.000000,120.883650,0.000000,1.000000,34.000000,0.000000,0.000000,0.000000,...,1.962253,4.000000,0.673603,0.337245,0.000000,0.237654,-0.469735,0.048251,8.268899,2.000000e-01
75%,2.000000,1.000000,6.000000,263.000000,0.000000,3.000000,51.000000,1.000000,1.000000,37.475050,...,2.354430,9.000000,1.204983,1.115771,1.180359,1.377016,0.630834,2.512776,10.361206,2.000000e-01
max,19.000000,13.000000,31.000000,2730.388400,11.000000,104.000000,116.000000,8.000000,22.000000,932.758670,...,5.251043,25.000000,68489.117953,9.671624,68493.313831,12.035948,6.997842,15.084366,16.934275,2.000000e-01


In [22]:
merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 828101 entries, 0 to 828100
Data columns (total 68 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   matchId                  828101 non-null  str    
 1   mapName                  828101 non-null  str    
 2   gameMode                 828101 non-null  str    
 3   createdAt                828101 non-null  str    
 4   DBNOs                    828101 non-null  int64  
 5   assists                  828101 non-null  int64  
 6   boosts_x                 828101 non-null  int64  
 7   damageDealt_x            828101 non-null  float64
 8   deathType                828101 non-null  str    
 9   headshotKills            828101 non-null  int64  
 10  heals_x                  828101 non-null  int64  
 11  killPlace                828101 non-null  int64  
 12  killStreaks              828101 non-null  int64  
 13  kills_x                  828101 non-null  int64  
 14  longestKill    